In [1]:
import argparse

import torch
from pfns.bar_distribution import FullSupportBarDistribution
from sklearn.metrics import r2_score

from tfmplayground.callbacks import ConsoleLoggerCallback
from tfmplayground.evaluation import get_openml_predictions, TOY_TASKS_REGRESSION
from tfmplayground.interface import NanoTabPFNRegressor
from tfmplayground.model import NanoTabPFNModel
from tfmplayground.priors import PriorDumpDataLoader
from tfmplayground.train import train
from tfmplayground.utils import get_default_device, set_randomness_seed, make_global_bucket_edges
import sys

# Simulate command-line arguments
sys.argv = ['pretrain_regression.py', '--epochs', '80', '--steps', '25', '--batchsize', '50']

parser = argparse.ArgumentParser()

parser.add_argument("--priordump", type=str, default="./50x3_1280k_regression.h5", help="path to the prior dump")
parser.add_argument("--saveweights", type=str, default="nanotabpfn_weights.pth", help="path to save the trained model to")
parser.add_argument("--savebuckets", type=str, default="nanotabpfn_buckets.pth", help="path to save the bucket edges to")
parser.add_argument("--heads", type=int, default=6, help="number of attention heads")
parser.add_argument("--embeddingsize", type=int, default=192, help="the size of the embeddings used for the cells")
parser.add_argument("--hiddensize", type=int, default=768, help="size of the hidden layer of the mlps")
parser.add_argument("--layers", type=int, default=6, help="number of transformer layers")
parser.add_argument("--batchsize", type=int, default=1, help="batch size used during training (before gradient accumulation)")
parser.add_argument("--accumulate", type=int, default=1, help="number of gradients to accumulate before updating the weights")
parser.add_argument("--lr", type=float, default=1e-4, help="learning rate")
parser.add_argument("--steps", type=int, default=100, help="number of steps that constitute one epoch (important for lr scheduler)")
parser.add_argument("--epochs", type=int, default=10000, help="number of epochs to train for")
parser.add_argument("--loadcheckpoint", type=str, default=None, help="checkpoint from which to continue training")
parser.add_argument("--n_buckets", type=int, default=100, help="number of buckets for the data loader")

args = parser.parse_args()


using GPU backend


/home/mzzhang/TFM-Playground/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
set_randomness_seed(2402)

device = get_default_device()
ckpt = None
if args.loadcheckpoint:
    ckpt = torch.load(args.loadcheckpoint)

prior = PriorDumpDataLoader(filename=args.priordump, num_steps=args.steps, batch_size=args.batchsize, device=device, starting_index=args.steps*(ckpt['epoch'] if ckpt else 0))

model = NanoTabPFNModel(
    num_attention_heads=args.heads,
    embedding_size=args.embeddingsize,
    mlp_hidden_size=args.hiddensize,
    num_layers=args.layers,
    num_outputs=args.n_buckets,
)

bucket_edges = make_global_bucket_edges(
    filename=args.priordump,
    n_buckets=args.n_buckets,
    device=device,
)

torch.save(
    bucket_edges,
    args.savebuckets,
)

if ckpt:
    model.load_state_dict(ckpt['model'])

dist = FullSupportBarDistribution(bucket_edges)

class EvaluationLoggerCallback(ConsoleLoggerCallback):
    def __init__(self, tasks):
        self.tasks = tasks

    def on_epoch_end(self, epoch: int, epoch_time: float, loss: float, model, **kwargs):
        regressor = NanoTabPFNRegressor(model, dist, device)
        predictions = get_openml_predictions(model=regressor, tasks=self.tasks)
        scores = []
        for dataset_name, (y_true, y_pred, _) in predictions.items():
            scores.append(r2_score(y_true, y_pred))
        avg_score = sum(scores) / len(scores)
        print(f'epoch {epoch:5d} | time {epoch_time:5.2f}s | mean loss {loss:5.2f} | avg r2 score {avg_score:.3f}',
              flush=True)


callbacks = [EvaluationLoggerCallback(TOY_TASKS_REGRESSION)]

trained_model, loss = train(
    model=model,
    prior=prior,
    criterion=dist,
    epochs=args.epochs,
    accumulate_gradients=args.accumulate,
    lr=args.lr,
    device=device,
    callbacks=callbacks,
    ckpt=ckpt
)

torch.save(trained_model.to('cpu').state_dict(), args.saveweights)

using GPU backend
./50x3_1280k_regression.h5
Using 5000000 y evals to estimate 100 buckets. Cut off the last 0 ys.


Could not download file from https://openml.org/datasets/0004/43959/dataset_43959.pq: HTTPSConnectionPool(host='www.openml.org', port=443): Max retries exceeded with url: https://www.openml.org/datasets/0004/43959/dataset_43959.pq (Caused by ResponseError('too many redirects'))


epoch     1 | time  0.98s | mean loss  1.54 | avg r2 score 0.003


Could not download file from https://openml.org/datasets/0004/43959/dataset_43959.pq: HTTPSConnectionPool(host='www.openml.org', port=443): Max retries exceeded with url: https://www.openml.org/datasets/0004/43959/dataset_43959.pq (Caused by ResponseError('too many redirects'))


epoch     2 | time  0.71s | mean loss  1.53 | avg r2 score 0.009


...

epoch    78 | time  0.69s | mean loss  0.57 | avg r2 score 0.500


Could not download file from https://openml.org/datasets/0004/43959/dataset_43959.pq: HTTPSConnectionPool(host='www.openml.org', port=443): Max retries exceeded with url: https://www.openml.org/datasets/0004/43959/dataset_43959.pq (Caused by ResponseError('too many redirects'))


epoch    79 | time  0.72s | mean loss  0.57 | avg r2 score 0.497


Could not download file from https://openml.org/datasets/0004/43959/dataset_43959.pq: HTTPSConnectionPool(host='www.openml.org', port=443): Max retries exceeded with url: https://www.openml.org/datasets/0004/43959/dataset_43959.pq (Caused by ResponseError('too many redirects'))


epoch    80 | time  0.72s | mean loss  0.58 | avg r2 score 0.495


In [1]:
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split

from tfmplayground import NanoTabPFNRegressor

# Load data
from sklearn.datasets import fetch_california_housing
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
import numpy as np

X, y = fetch_california_housing(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

model = NanoTabPFNRegressor()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(r2_score(y_test, y_pred))

using GPU backend
using GPU backend


RuntimeError: Error(s) in loading state_dict for NanoTabPFNModel:
	Missing key(s) in state_dict: "transformer_encoder.transformer_blocks.0.external_gate", "transformer_encoder.transformer_blocks.0.self_attention_between_datapoints.core._w_out", "transformer_encoder.transformer_blocks.0.self_attention_between_datapoints.core._w_qkv", "transformer_encoder.transformer_blocks.0.self_attention_between_features.core._w_out", "transformer_encoder.transformer_blocks.0.self_attention_between_features.core._w_qkv", "transformer_encoder.transformer_blocks.1.external_gate", "transformer_encoder.transformer_blocks.1.self_attention_between_datapoints.core._w_out", "transformer_encoder.transformer_blocks.1.self_attention_between_datapoints.core._w_qkv", "transformer_encoder.transformer_blocks.1.self_attention_between_features.core._w_out", "transformer_encoder.transformer_blocks.1.self_attention_between_features.core._w_qkv", "transformer_encoder.transformer_blocks.2.external_gate", "transformer_enco